# Feature Engineering – Practical Machine Learning Series

## Learning objectives
By the end of this notebook, you should be able to explain the concept, identify appropriate techniques, implement them with pandas/scikit-learn, and recognize common risks.

**Important:** Feature engineering must be learned from the training data only when a transformation learns statistics or target relationships. Fit transformations on training data and apply the fitted transformation to validation/test data.

## 1. What is a feature?

A **feature** is an input attribute used by a machine-learning model to make a prediction. For a customer-churn model, examples include monthly charges, contract type, tenure, and number of support calls.

A **variable** is a general statistical/programming term for a quantity that can vary. A variable becomes a feature when it is used as model input.

- **Feature (X):** information supplied to the model.
- **Target (y):** the outcome the model is expected to predict.
- **Record/observation:** one row, such as one customer.

A column can be a feature in one problem and a target in another. Its role depends on the prediction task.

In [ ]:
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105],
    "monthly_charges": [800, 1200, 950, 1500, 700],
    "tenure_months": [2, 18, 7, 30, 4],
    "support_calls": [5, 1, 3, 0, 6],
    "contract_type": ["Monthly", "Annual", "Monthly", "Annual", "Monthly"],
    "churned": [1, 0, 0, 0, 1]
})
X = customers.drop(columns=["churned", "customer_id"])
y = customers["churned"]
display(customers)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

## 2. What is feature engineering and why is it important?

**Feature engineering** is the process of creating, transforming, encoding, or selecting input features so that useful patterns become easier for a model to learn.

It matters because:
- raw data may not express the business concept directly;
- models often work better with suitable numeric representations;
- useful combinations can reveal relationships that individual columns hide;
- irrelevant, noisy, or leaking columns can harm generalization.

Feature engineering is not a guarantee of better performance. Every change should be validated against a baseline using an appropriate validation strategy.

## 3. Domain knowledge

Domain knowledge helps determine whether a feature is meaningful, valid, and available at prediction time.

Examples:
- Insurance: claim amount divided by insured value may indicate claim severity.
- Retail: days since last purchase can describe customer recency.
- HR: employee tenure can be calculated from joining date.
- Healthcare: a measurement must be interpreted using clinical context and units.

Check units, business rules, timing, and whether a feature would genuinely be known when the prediction is made.

## 4. Good features vs bad features

| Good feature | Potentially bad feature |
|---|---|
| Available at prediction time | Only known after the outcome |
| Consistent definition and units | Ambiguous or inconsistent meaning |
| Relevant to the problem | Irrelevant noise |
| Validated on unseen data | Chosen only because it improves training score |
| No target information embedded | Target leakage |

A high correlation with the target does not automatically make a feature valid.

## 5. Feature engineering workflow

1. Define the prediction target and prediction time.
2. Understand columns, units, data quality, and domain rules.
3. Create a simple baseline model.
4. Create candidate features using training data.
5. Encode or transform features as needed.
6. Select useful features without using held-out test information.
7. Validate with suitable metrics and cross-validation.
8. Document definitions and deploy the same transformations consistently.

## 6. Feature creation, transformation, selection, and extraction

- **Creation:** combine existing columns, e.g. `total_spend = price * quantity`.
- **Transformation:** change a feature's scale or distribution, e.g. `log1p(income)`.
- **Selection:** retain a subset of existing features, e.g. remove redundant or irrelevant columns.
- **Extraction:** derive a new representation, e.g. PCA components from many correlated numeric columns.

These terms are related but not interchangeable. Selection keeps chosen columns; extraction creates a transformed representation.

In [ ]:
# Feature creation
customers["monthly_support_rate"] = (
    customers["support_calls"] / customers["tenure_months"].clip(lower=1)
)

# A simple transformation
customers["log_monthly_charges"] = __import__("numpy").log1p(customers["monthly_charges"])

display(customers[[
    "monthly_charges", "tenure_months", "support_calls",
    "monthly_support_rate", "log_monthly_charges"
]])

## 7. Feature leakage

**Feature leakage** occurs when a model uses information that would not legitimately be available at the time a prediction is made, or when information from validation/test data influences training.

Examples:
- using a claim settlement date to predict whether the claim will be approved;
- using a cancellation reason to predict future customer churn;
- calculating imputation/scaling statistics on the complete dataset before splitting;
- target encoding categories using validation/test target values.

Prevent leakage by defining prediction time, splitting data first, fitting learned preprocessing only on training data, and using time-based splits when the real task predicts the future.

In [ ]:
# Demonstration: split first, then fit preprocessing only on training data.
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_demo = customers[["monthly_charges", "tenure_months", "support_calls"]]
y_demo = customers["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X_demo, y_demo, test_size=0.4, random_state=42, stratify=y_demo
)

numeric_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)
X_train_ready = numeric_pipeline.fit_transform(X_train)
X_test_ready = numeric_pipeline.transform(X_test)

print("Train transformed shape:", X_train_ready.shape)
print("Test transformed shape:", X_test_ready.shape)

## 8. Feature engineering in the ML pipeline

Typical order:

`Raw data → validation/cleaning → train/validation split → feature engineering & preprocessing → model training → evaluation → deployment`

Some deterministic, row-wise transformations can be calculated before splitting if they do not use target or dataset-wide statistics. However, the safest general practice is to put learned preprocessing inside a scikit-learn `Pipeline` or `ColumnTransformer`.

 State its formula, business meaning, and whether it is available at prediction time.